# 1. Import the necessary libraries

In [ ]:
from problems import get_problem
from runners import run_mesh, run_mesh_old, run_nsga2

from functools import partial
from joblib import Parallel, delayed

import contextlib
import io

# 2. Define the general configuration of the experiments

In [ ]:
#################### CUSTOMIZABLE ####################

# Number of processes to execute the experiments in parallel
workers = 4

# Objective function configuration (function name, number of objectives, number of decision variables)
experiments = [('zdt2', 2, 10), ('zdt4', 2, 10), ('dtlz1', 3, 10), ('dtlz3', 3, 10)]

# Execution configuration
num_runs = 30 # Number of runs
max_fitness_eval = 15000 # Maximum fitness evaluations (not used if it is None)
population_size = 100 # Population size
random_state = None # Set a seed for random numbers (not used if it is None)

results_folder = f'results' # Folder to store the results
tuning_folder = f'hyperparams' # Folder to get the tuned parameters
######################################################

# 3 Define auxiliar functions

In [ ]:
# Microgrid objetive function
# position_min_value = np.array([10, 1, 50]) # Lower bound of problem [max PV generation, number of wind turbines, battery capacity]
# position_max_value = np.array([450, 5, 500]) # Upper bound of problem [max PV generation, number of wind turbines, battery capacity]
# def func(args):
#     r = techno_ka(args[0], args[1], 0.8, args[2], select_bat, solar_data, wind_data, load_ind)[:objective_dim]
#     #r = techno_ka(args[0], args[1], 0.8, args[2], select_bat, solar_data, wind_data, load_ind)[1:3]
#     r[-1] = -r[-1] # Maximizing renewable factor
#     return r

# Function to get the fitness function configuration
def get_fitness_function(func_name, objective_dim, position_dim):
	func, position_min_value, position_max_value = get_problem(func_name, n_var=position_dim, n_obj=objective_dim)
	return func, objective_dim, position_dim, position_min_value, position_max_value

# Function to capture errors during parallel execution without stopping the other processes
def safe_run(run_function, params):
	try:
		f = io.StringIO()
		with contextlib.redirect_stdout(f):
			status = run_function(*params)
		return status, f.getvalue()
	except Exception as e:
		return f'Error: {str(e)}'

# 4. Run experiments with the algorithms (in parallel)

## 4.1 Run MESH

In [ ]:
#################### CUSTOMIZABLE ####################

# MESH fixed parameters
mesh_memory_size = population_size # Maximum number of particles in memory
mesh_global_best_type = [0] # 0 -> Sigma method (G1) | 1 -> Sigma method in fronts (G2)
mesh_dm_pool_type = [0] # 0 -> Sampling from memory (S1) | 1 -> Sampling from population (S2) | 2 -> Sampling from memory and population (S3)
mesh_differential_evolution_type = [0] # 0 -> DE\rand\1\Bin (D1) | 1 -> DE\rand\2\Bin (D2) | 2 -> DE/Best/1/Bin (D3) | 3 -> DE/Current-to-best/1/Bin (D4) | 4 -> DE/Current-to-rand/1/Bin (D5)

# MESH tunable parameters
# OBS: The function "run_mesh" and "run_mesh_old" will select automatically the tuned parameters if they exists ("hyperparams" folder generated by "fine_tuning.ipynb")
mesh_communication_probability = 0.8 # Communication probability
mesh_mutation_rate = 0.9 # Mutation rate
mesh_personal_guide_array_size = 1 # Number of personal guides
######################################################

In [ ]:
# Set the list of parameters
params_list = [
  	[(F'MESH_G{gb_type+1}S{pool_type+1}D{de_type+1}_{exp[0]}_{exp[1]}_{exp[2]}', results_folder, tuning_folder, num_runs, max_fitness_eval, population_size, random_state),
	 get_fitness_function(exp[0], exp[1], exp[2]),
	 (mesh_memory_size, gb_type, pool_type, de_type),
	 (mesh_communication_probability, mesh_mutation_rate, mesh_personal_guide_array_size)]
     for exp in experiments
     for gb_type in mesh_global_best_type
	 for pool_type in mesh_dm_pool_type
	 for de_type in mesh_differential_evolution_type
]

# Execute in parallel
Parallel(n_jobs=workers)(delayed(safe_run)(run_mesh, params) for params in params_list)

### Run MESH (previous implementation)

In [ ]:
# Set the list of parameters
params_list = [
  	[(F'OLD_MESH_G{gb_type+1}S{pool_type+1}D{de_type+1}_{exp[0]}_{exp[1]}_{exp[2]}', results_folder, tuning_folder, num_runs, max_fitness_eval, population_size, random_state),
	 get_fitness_function(exp[0], exp[1], exp[2]),
	 (mesh_memory_size, gb_type, pool_type, de_type),
	 (mesh_communication_probability, mesh_mutation_rate, mesh_personal_guide_array_size)]
     for exp in experiments
     for gb_type in mesh_global_best_type
	 for pool_type in mesh_dm_pool_type
	 for de_type in mesh_differential_evolution_type
]

# Execute in parallel
Parallel(n_jobs=workers)(delayed(safe_run)(run_mesh_old, params) for params in params_list)

# 4.2 Run NSGA-II

In [ ]:
#################### CUSTOMIZABLE ####################

# NSGA2 tunable parameters
# OBS: The function "run_nsga2" will select automatically the tuned parameters if they exists ("hyperparams" folder generated by "fine_tuning.ipynb")
nsga2_recombination_probability = 0.45
nsga2_eta_recombination = 19
nsga2_mutation_probability = 0.06
nsga2_eta_mutation = 14.7
######################################################

# Set the list of parameters
params_list = [
  	[(F'NSGA2_{exp[0]}_{exp[1]}_{exp[2]}', results_folder, tuning_folder, num_runs, max_fitness_eval, population_size, random_state),
	 get_fitness_function(exp[0], exp[1], exp[2]),
	 (),
	 (nsga2_recombination_probability, nsga2_eta_recombination, nsga2_mutation_probability, nsga2_eta_mutation)]
     for exp in experiments
]

# Execute in parallel
Parallel(n_jobs=workers)(delayed(safe_run)(run_nsga2, params) for params in params_list)